In [1]:
import os
import glob
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 192
BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-3

TRAIN_DIR = "/kaggle/input/pixel-play-26/Avenue_Corrupted-20251221T112159Z-3-001/Avenue_Corrupted/Dataset/training_videos"
TEST_DIR  = "/kaggle/input/pixel-play-26/Avenue_Corrupted-20251221T112159Z-3-001/Avenue_Corrupted/Dataset/testing_videos"
TEMPLATE_CSV = "/kaggle/input/sample/submission-8.csv"

MODEL_PATH = "baseline_cae.pth"
SUBMISSION_PATH = "submission_baseline.csv"

In [3]:
def unflip_if_needed(img):
    h = img.shape[0]
    top_mean = img[:h//2].mean()
    bottom_mean = img[h//2:].mean()

    # If top is brighter than bottom, image is likely flipped
    if top_mean > bottom_mean:
        img = np.flipud(img).copy()

    return img

In [4]:
def preprocess_frame(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.bilateralFilter(img, 5, 75, 75)
    img = img.astype(np.float32) / 255.0
    img = unflip_if_needed(img)
    return img

In [5]:
class AvenueFrameDataset(Dataset):
    def __init__(self, root):
        self.frames = []

        for vid in sorted(os.listdir(root)):
            paths = sorted(glob.glob(os.path.join(root, vid, "*.jpg")))
            for p in paths:
                self.frames.append(preprocess_frame(p))

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        x = self.frames[idx]
        x = torch.tensor(x).unsqueeze(0)  # (1, H, W)
        return x

In [6]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

In [7]:
def train():
    dataset = AvenueFrameDataset(TRAIN_DIR)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8)

    model = ConvAutoencoder().to(DEVICE)
    model = nn.DataParallel(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for x in tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            x = x.to(DEVICE)

            recon = model(x)
            loss = loss_fn(recon, x)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.6f}")

    torch.save(model.state_dict(), MODEL_PATH)
    print("Baseline CAE model saved.")

if __name__ == "__main__":
        train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Epoch 1/5:   0%|          | 0/1151 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Epoch 1/5: 100%|██████████| 1151/1151 [

Epoch 1 | Loss: 0.000876


Epoch 2/5: 100%|██████████| 1151/1151 [00:15<00:00, 72.10it/s]


Epoch 2 | Loss: 0.000179


Epoch 3/5: 100%|██████████| 1151/1151 [00:15<00:00, 72.29it/s]


Epoch 3 | Loss: 0.000143


Epoch 4/5: 100%|██████████| 1151/1151 [00:15<00:00, 72.93it/s]


Epoch 4 | Loss: 0.000126


Epoch 5/5: 100%|██████████| 1151/1151 [00:16<00:00, 71.64it/s]

Epoch 5 | Loss: 0.000116
Baseline CAE model saved.


In [26]:
def extract_frame_number(path):
    return int(os.path.basename(path).split("_")[1].split(".")[0])

In [33]:
def generate_submission():
    sample = pd.read_csv(TEMPLATE_CSV)
    sample["Predicted"] = 0.0

    state = torch.load("/kaggle/working/baseline_cae.pth", map_location=DEVICE)
    
    
    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", ""): v for k, v in state.items()}
    
    model = ConvAutoencoder().to(DEVICE)
    model.load_state_dict(state)
    model.eval()

    score_map = {}
    all_scores = []

    with torch.no_grad():
        for vid in tqdm(sorted(os.listdir(TEST_DIR)), desc="Inference"):
            paths = sorted(glob.glob(os.path.join(TEST_DIR, vid, "*.jpg")))
            frames = [preprocess_frame(p) for p in paths]
            frame_nums = [extract_frame_number(p) for p in paths]

            for img, fn in zip(frames, frame_nums):
                x = torch.tensor(img).unsqueeze(0).unsqueeze(0).to(DEVICE)
                recon = model(x)
                err = torch.mean((recon - x) ** 2).item()

                fid = f"{int(vid)}_{fn}"
                score_map[fid] = err
                all_scores.append(err)

    # Global normalization
    mn, mx = min(all_scores), max(all_scores)
    for k in score_map:
        score_map[k] = (score_map[k] - mn) / (mx - mn + 1e-8)

    for i in range(len(sample)):
        fid = sample.at[i, "Id"]
        if fid in score_map:
            sample.at[i, "Predicted"] = score_map[fid]

    sample.to_csv(SUBMISSION_PATH, index=False)
    print("submission_baseline.csv generated.")

In [34]:
generate_submission()

Inference: 100%|██████████| 21/21 [02:05<00:00,  5.99s/it]


submission_baseline.csv generated.
